In [7]:
import pandas as pd
import numpy as np

# تواريخ
dates = pd.date_range(start="2024-01-01", periods=120)

data = []

# السلع
products = ["tomato", "potato", "rice", "sugar", "wheat"]

for product in products:
    # سعر ابتدائي مختلف لكل سلعة
    base_price = {
        "tomato": 10,
        "potato": 8,
        "rice": 15,
        "sugar": 12,
        "wheat": 20
    }

    price = base_price[product]

    for date in dates:
        # تغيير عشوائي بسيط
        change = np.random.randn() * 0.5
        price += change

        # simulate زيادة مفاجئة (أزمة)
        if np.random.rand() < 0.05:
            price += np.random.rand() * 5

        # simulate نقص (هبوط مفاجئ)
        if np.random.rand() < 0.03:
            price -= np.random.rand() * 3

        data.append([date, product, round(price, 2)])

# DataFrame
df = pd.DataFrame(data, columns=["date", "product", "price"])

# حفظ الملف
df.to_csv("food_prices.csv", index=False)

print("✅ Data generated successfully!")

✅ Data generated successfully!


In [8]:
import os
print("Current directory:", os.getcwd())

Current directory: C:\Users\acer\PyCharmMiscProject


In [11]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Generate Data
# -----------------------------

dates = pd.date_range(start="2024-01-01", periods=120)

data = []

products = ["tomato", "potato", "rice", "sugar", "wheat"]

for product in products:
    base_price = {
        "tomato": 10,
        "potato": 8,
        "rice": 15,
        "sugar": 12,
        "wheat": 20
    }

    price = base_price[product]

    for date in dates:
        change = np.random.randn() * 0.5
        price += change

        # sudden increase
        if np.random.rand() < 0.05:
            price += np.random.rand() * 5

        # sudden decrease
        if np.random.rand() < 0.03:
            price -= np.random.rand() * 3

        data.append([date, product, round(price, 2)])

df = pd.DataFrame(data, columns=["date", "product", "price"])
df.to_csv("food_prices.csv", index=False)

print("✅ Data generated successfully!")

# -----------------------------
# 2. Load Data
# -----------------------------

df = pd.read_csv("food_prices.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["product", "date"])

# -----------------------------
# 3. Feature Engineering
# -----------------------------

df["prev_price"] = df.groupby("product")["price"].shift(1)

df["price_change"] = (df["price"] - df["prev_price"]) / df["prev_price"]

df["moving_avg"] = df.groupby("product")["price"].rolling(7).mean().reset_index(0, drop=True)

# -----------------------------
# 4. Alert Logic
# -----------------------------

def detect_alert(row):
    if pd.isna(row["price_change"]):
        return "🔄 No Data"

    if row["price_change"] > 0.1:
        return "🚨 Strong Increase"
    elif row["price_change"] > 0.05:
        return "⚠️ Increase"
    elif row["price"] > row["moving_avg"]:
        return "📈 Uptrend"
    else:
        return "✅ Stable"

df["alert"] = df.apply(detect_alert, axis=1)

# -----------------------------
# 5. Forecast (بعد التصحيح)
# -----------------------------

def calculate_trend(group):
    group = group.sort_values("date")
    group["trend"] = group["price"].diff()
    return group

df = df.groupby("product").apply(calculate_trend)

# ✅ حل المشكلة هنا
df = df.reset_index(drop=True)

# حساب متوسط الاتجاه
df["avg_trend"] = df.groupby("product")["trend"].rolling(5).mean().reset_index(0, drop=True)

# Forecast بعد 3 أيام
def forecast_price(row):
    if pd.isna(row["avg_trend"]):
        return row["price"]
    return row["price"] + (row["avg_trend"] * 3)

df["forecast_3days"] = df.apply(forecast_price, axis=1)

# Forecast Alert
def forecast_alert(row):
    if pd.isna(row["forecast_3days"]):
        return "No Forecast"

    change = (row["forecast_3days"] - row["price"]) / row["price"]

    if change > 0.15:
        return "🚨 High Risk Increase"
    elif change > 0.07:
        return "⚠️ المتوقع ارتفاع"
    elif change < -0.1:
        return "📉 المتوقع انخفاض"
    else:
        return "✅ مستقر"

df["forecast_alert"] = df.apply(forecast_alert, axis=1)

# -----------------------------
# 6. Save Final File
# -----------------------------

df.to_csv("food_prices_with_forecast.csv", index=False)

print("✅ Forecast generated and saved successfully!")

✅ Data generated successfully!
✅ Forecast generated and saved successfully!


C:\Users\acer\AppData\Local\Temp\ipykernel_9688\3294841634.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("product").apply(calculate_trend)


In [12]:
def calculate_risk(row):
    score = 0

    # 1. التغير الحالي
    if row["price_change"] > 0.1:
        score += 40
    elif row["price_change"] > 0.05:
        score += 25

    # 2. التوقع
    change = (row["forecast_3days"] - row["price"]) / row["price"]

    if change > 0.15:
        score += 40
    elif change > 0.07:
        score += 25

    # 3. الاتجاه
    if row["avg_trend"] > 0:
        score += 20

    return min(score, 100)

df["risk_score"] = df.apply(calculate_risk, axis=1)
def risk_level(score):
    if score >= 70:
        return "🔴 High Risk"
    elif score >= 40:
        return "🟡 Medium Risk"
    else:
        return "🟢 Low Risk"

df["risk_level"] = df["risk_score"].apply(risk_level)
df.to_csv("food_prices_final.csv", index=False)